# MEGC walrasiano: shock del datacenter Stargate Argentina

Modelo de Equilibrio General Computable de 3 sectores para una economía pequeña y abierta.
Simula la instalación de un datacenter de IA de 500 MW (CapEx USD 18.900 M bajo RIGI).

La lógica del modelo vive en `stargate_cge.ipynb` (misma carpeta); este notebook muestra las ecuaciones, la calibración y los resultados.

## 1. Estructura

| Sector | Tecnología | Precio |
|---|---|---|
| **E** Energía | Cobb-Douglas en $L$, $K$ (participación del trabajo 20%) | $p_E$ endógeno, no transable |
| **D** Datacenter | Leontief en energía, trabajo y hardware importado; capacidad fija por el CapEx | $p_D^*$ mundial; hardware a $P_H^*=1$ constante |
| **R** Resto | Cobb-Douglas en $L$, $K$, $E$; intensivo en trabajo | $p_R=1$ transable (numerario) |

**Cierres:** *corto plazo* = capital específico por sector ($r_E \neq r_R$); *largo plazo* = capital doméstico móvil entre E y R ($r_E = r_R$).

**Parámetros fijos:** CapEx 18,9 bn · 500 MW · 4,55 TWh/año · USD 305 M/año → $p_{E0} = 305/4{,}55 = 67{,}0$ USD/MWh.

**Supuestos de calibración (ajustables):** PBI USD 640 bn; 140 TWh (55% empresas, 45% hogares); participación del trabajo en el PBI 50%; elasticidad de oferta laboral 0,2; datacenter con ~1.500 empleos (USD 60 M de masa salarial), $r^*+\delta = 25\%$, ingreso = 1,2 × costo, Ganancias RIGI 25%.

## 2. Sistema de ecuaciones

**Firmas (maximización de beneficios)**

$$
\begin{aligned}
&\textbf{Energía:}\quad \max_{L_E,K_E}\; \pi_E = p_E X_E - wL_E - r_E K_E \quad\text{s.a.}\quad X_E = A_E L_E^{\alpha}K_E^{1-\alpha}\\
&\qquad w = \alpha\,\frac{p_E X_E}{L_E},\qquad r_E = (1-\alpha)\frac{p_E X_E}{K_E}
\;\Longrightarrow\; p_E = c_E(w,r_E) = \frac{1}{A_E}\Big(\frac{w}{\alpha}\Big)^{\alpha}\Big(\frac{r_E}{1-\alpha}\Big)^{1-\alpha}\\[6pt]
&\textbf{Resto:}\quad \max\; \pi_R = X_R - wL_R - r_R K_R - p_E E_R \quad\text{s.a.}\quad X_R = A_R L_R^{\theta_L}K_R^{\theta_K}E_R^{\theta_E},\;\; \textstyle\sum\theta=1\\
&\qquad wL_R=\theta_L X_R,\quad r_R K_R=\theta_K X_R,\quad p_E E_R=\theta_E X_R
\;\Longrightarrow\; 1 = c_R(w,r_R,p_E)\\[6pt]
&\textbf{Datacenter:}\quad \max\; \pi_D = p_D^* X_D - p_E E_D - wL_D - (r^*+\delta)P_H^* K_D\\
&\qquad X_D=\min\Big\{\tfrac{E_D}{a_{ED}},\tfrac{L_D}{a_{LD}},\tfrac{K_D}{a_{KD}}\Big\},\quad K_D=\bar K_D^{*}=18.9
\;\Rightarrow\; \bar X_D = 500\text{ MW},\; E_D = 4.55\text{ TWh},\; L_D=a_{LD}\bar X_D
\end{aligned}
$$

**Hogar representativo (maximización de utilidad)**

$$
\max_{C_R,C_E,\ell}\; U = C_R^{\mu_R}C_E^{\mu_E}\ell^{\mu_\ell}
\quad\text{s.a.}\quad C_R + p_E C_E + w\ell = F \equiv wT + r_E\bar K_E + r_R\bar K_R + \tau\,\pi_D
$$
$$
C_R=\mu_R F,\qquad C_E=\frac{\mu_E F}{p_E},\qquad \ell=\frac{\mu_\ell F}{w},\qquad L^s = T-\ell
$$

**Vaciado de mercados**

$$
\begin{aligned}
&\text{Eléctrico:} && X_E = E_R + C_E + E_D\\
&\text{Laboral:} && L_E + L_R + L_D = T - \ell\\
&\text{Capital (CP):} && K_E=\bar K_E,\quad K_R=\bar K_R\\
&\text{Capital (LP):} && K_E+K_R=\bar K,\quad r_E=r_R\\
&\text{Capital extranjero:} && K_D=\bar K_D^{*}\ \text{a } r^*+\delta \text{ dado}\\
&\text{Transable (Walras):} && C_R - X_R = p_E E_D + wL_D + \tau\pi_D
\end{aligned}
$$

Incógnitas $(p_E, w, r_E, r_R, X_E, X_R)$: 6 ecuaciones (2 de beneficio cero, eléctrico, laboral, 2 de capital).

In [1]:
# Carga del modelo: stargate_cge.ipynb tiene que estar en la misma carpeta que este notebook.
# En Google Colab cada notebook se abre solo (en /content), así que si falta se pide subirlo.
from pathlib import Path

MODELO = Path("stargate_cge.ipynb")
if not MODELO.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            f"No se encontró {MODELO.resolve()}. Abrí este notebook desde la carpeta que contiene stargate_cge.ipynb."
        ) from None
    print("No se encontró stargate_cge.ipynb. Subilo con el botón de abajo:")
    subidos = [n for n in files.upload() if n.endswith(".ipynb")]
    if not subidos:
        raise FileNotFoundError("No se subió ningún .ipynb con el modelo.")
    Path(subidos[0]).rename(MODELO)  # Colab renombra duplicados como "stargate_cge (1).ipynb"

%run stargate_cge.ipynb

In [2]:
import pandas as pd


pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_rows", 50)

## 3. Calibración

In [3]:
eco, dc = EconomiaBase(), ShockDatacenter()
cal = calibrar(eco, dc)

print(f"Precio base de la energía: {1000 * dc.precio_energia_base:.2f} USD/MWh")
pd.Series({k: v for k, v in cal.items()}, name="parámetro calibrado").to_frame()

Precio base de la energía: 67.03 USD/MWh


,parámetro calibrado
pE0,0.0670
XE0,140.0000
XR0,635.7769
KE0,7.5077
KR0,312.4923
T,480.0000
aL,0.2000
AE,24.6056
thL,0.5004
thK,0.4915


## 4. Equilibrio antes y después del shock

In [4]:
escenarios = {
    "Base": resolver(cal, "largo", con_dc=False),
    "Corto plazo": resolver(cal, "corto", con_dc=True),
    "Largo plazo": resolver(cal, "largo", con_dc=True),
}
resultados = tabla_resultados(cal, escenarios)
resultados

,Base,Corto plazo,Largo plazo,Δ% Corto plazo,Δ% Largo plazo
p_E (USD/MWh),67.0330,68.8137,67.0369,2.6565,0.0059
"w (salario, base=1)",1.0000,1.0001,0.9999,0.0075,-0.0097
r_E (base=1),1.0000,1.0333,1.0001,3.3296,0.0098
r_R (base=1),1.0000,0.9995,1.0001,-0.0510,0.0098
X_E generación (TWh),140.0000,140.9180,144.5086,0.6557,3.2205
E_R resto (TWh),77.0000,74.9692,76.9435,-2.6374,-0.0734
C_E hogares (TWh),63.0000,61.3988,63.0151,-2.5416,0.0240
E_D datacenter (TWh),0.0000,4.5500,4.5500,NaN,NaN
X_R resto (USD bn),635.7769,635.4528,635.3476,-0.0510,-0.0675
L_E,1.8769,1.9393,1.9377,3.3218,3.2365


In [5]:
# Descomposición de los 4,55 TWh del datacenter en el corto plazo
b0, b1 = escenarios["Base"], escenarios["Corto plazo"]
pd.Series({
    "Nueva generación (TWh)": b1["XE"] - b0["XE"],
    "Desplazamiento resto E_R (TWh)": b0["ER"] - b1["ER"],
    "Desplazamiento hogares C_E (TWh)": b0["CE"] - b1["CE"],
    "Total = E_D (TWh)": b1["ED"],
}, name="corto plazo").to_frame()

,corto plazo
Nueva generación (TWh),0.9180
Desplazamiento resto E_R (TWh),2.0308
Desplazamiento hogares C_E (TWh),1.6012
Total = E_D (TWh),4.5500


## 5. Sensibilidad: bienestar según el margen del datacenter y la alícuota RIGI

In [6]:
from dataclasses import replace

filas = {}
for markup in [1.0, 1.1, 1.2, 1.4]:
    for alicuota in [0.0, 0.15, 0.25, 0.35]:
        c = calibrar(eco, replace(dc, markup=markup, alicuota_rigi=alicuota))
        t = tabla_resultados(c, {"Base": resolver(c, "largo", False), "LP": resolver(c, "largo", True)})
        filas[(markup, alicuota)] = t.loc["Variación equivalente (USD bn)", "LP"]

pd.Series(filas).unstack().rename_axis(index="markup", columns="alícuota RIGI")

alícuota RIGI,0.0000,0.1500,0.2500,0.3500
markup,,,,
1.0000,0.0000,0.0000,0.0000,0.0000
1.1000,0.0000,0.0764,0.1273,0.1782
1.2000,0.0000,0.1527,0.2545,0.3563
1.4000,0.0000,0.3054,0.5090,0.7126


## 6. Lectura

- **Corto plazo:** con el capital eléctrico fijo, la generación casi no responde; la mayor parte de los 4,55 TWh se obtiene desplazando consumo de hogares y del resto vía suba de $p_E$. La renta va al capital de generación ($r_E$ ↑).
- **Largo plazo:** el capital migra hacia E y $p_E$ vuelve prácticamente a su nivel base (insensibilidad de precios de factores en una economía pequeña con $p_R$ dado; Rybczynski contrae levemente R).
- **PBI ≫ ingreso nacional:** casi todo el valor agregado del datacenter es retorno del hardware extranjero. El país retiene salarios, factura eléctrica y Ganancias, que es exactamente el déficit comercial en R que el proyecto financia (chequeo de Walras).
- **Limitaciones:** sin fase de construcción, sin gas importado ni importación de electricidad, sin generación propia de Sur Energy.